# Baby Shower Problem in Pure Python
## Five Exact Methods for Book Problem 2.14


## Problem Statement

Book problem `2.14` is a binary invitation problem for a baby shower. Each guest is associated with
the value of the gift they would bring, and the invitation list must satisfy several logical rules:

- José attends only if Valentina attends,
- José does not attend if both Ana and Daniel attend,
- Ana and Camila are complements: exactly one of them attends,
- Ana attends only if Daniel and Valentina attend together.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


NAMES = {1: "Jose", 2: "Ana", 3: "Daniel", 4: "Valentina", 5: "Camila"}
GIFT_VALUE = {1: 62, 2: 30, 3: 85, 4: 82, 5: 77}
EXPECTED = {"invited": [1, 4, 5], "names": ["Jose", "Valentina", "Camila"], "total_value": 221}


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["total_value"], tuple(left["invited"]))
    right_key = (right["total_value"], tuple(right["invited"]))
    return right if right_key > left_key else left


def build_solution(invited):
    invited = sorted(invited)
    x = {guest: int(guest in invited) for guest in NAMES}
    if x[1] > x[4]:
        return None
    if x[1] > 2 - x[2] - x[3]:
        return None
    if x[2] != 1 - x[5]:
        return None
    if x[4] + x[3] > 1 + x[2]:
        return None
    if x[2] > x[4]:
        return None
    if x[2] > x[3]:
        return None
    return {"invited": invited, "names": [NAMES[guest] for guest in invited], "total_value": sum(GIFT_VALUE[guest] for guest in invited)}


@timed
def solve_baby_naive():
    best = None
    for mask in range(1 << len(NAMES)):
        invited = [guest for guest in NAMES if (mask >> (guest - 1)) & 1]
        best = choose_better(best, build_solution(invited))
    return best


@timed
def solve_baby_backtracking():
    best = None
    ordered_guests = sorted(NAMES, key=lambda guest: GIFT_VALUE[guest], reverse=True)

    def branch(index, invited, current_value):
        nonlocal best
        remaining_value = sum(GIFT_VALUE[guest] for guest in ordered_guests[index:])
        if best is not None and current_value + remaining_value <= best["total_value"]:
            return
        if index == len(ordered_guests):
            best = choose_better(best, build_solution(invited))
            return
        guest = ordered_guests[index]
        branch(index + 1, invited, current_value)
        branch(index + 1, invited + [guest], current_value + GIFT_VALUE[guest])

    branch(0, [], 0)
    return best


@timed
def solve_baby_reduced_search():
    case_ana = build_solution([2, 3, 4])
    case_camila = choose_better(build_solution([4, 5]), build_solution([1, 4, 5]))
    return choose_better(case_ana, case_camila)


@timed
def solve_baby_dynamic_programming():
    @lru_cache(maxsize=None)
    def dp(index, chosen_mask):
        if index == len(NAMES):
            invited = [guest for guest in NAMES if (chosen_mask >> (guest - 1)) & 1]
            candidate = build_solution(invited)
            if candidate is None:
                return None
            return (candidate["total_value"], tuple(candidate["invited"]))
        guest = index + 1
        best = dp(index + 1, chosen_mask)
        with_guest = dp(index + 1, chosen_mask | (1 << (guest - 1)))
        if with_guest is not None and (best is None or with_guest > best):
            best = with_guest
        return best

    packed = dp(0, 0)
    return build_solution(packed[1])


@timed
def solve_baby_branch_and_bound():
    best = solve_baby_reduced_search()[0]
    ordered_guests = sorted(NAMES, key=lambda guest: GIFT_VALUE[guest], reverse=True)

    def branch(index, invited, current_value):
        nonlocal best
        remaining_value = sum(GIFT_VALUE[guest] for guest in ordered_guests[index:])
        if current_value + remaining_value <= best["total_value"]:
            return
        if index == len(ordered_guests):
            best = choose_better(best, build_solution(invited))
            return
        guest = ordered_guests[index]
        branch(index + 1, invited + [guest], current_value + GIFT_VALUE[guest])
        branch(index + 1, invited, current_value)

    branch(0, [], 0)
    return best


## 1. Naive Enumeration


In [2]:
naive_result, naive_time = solve_baby_naive()
print("Naive result:", naive_result)
print(f"Elapsed time: {naive_time:.6f} seconds")
assert naive_result == EXPECTED


Naive result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.000046 seconds


## 2. Backtracking With Pruning


In [3]:
backtracking_result, backtracking_time = solve_baby_backtracking()
print("Backtracking result:", backtracking_result)
print(f"Elapsed time: {backtracking_time:.6f} seconds")
assert backtracking_result == EXPECTED


Backtracking result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.000045 seconds


## 3. Constraint-Driven Reduced Search


In [4]:
print("Exact reduction: x_Ana = 1 - x_Camila, so there are only two structural cases to compare.")
reduced_result, reduced_time = solve_baby_reduced_search()
print("Reduced-search result:", reduced_result)
print(f"Elapsed time: {reduced_time:.6f} seconds")
assert reduced_result == EXPECTED


Exact reduction: x_Ana = 1 - x_Camila, so there are only two structural cases to compare.
Reduced-search result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.000009 seconds


## 4. Dynamic Programming


In [5]:
dp_result, dp_time = solve_baby_dynamic_programming()
print("Dynamic-programming result:", dp_result)
print(f"Elapsed time: {dp_time:.6f} seconds")
assert dp_result == EXPECTED


Dynamic-programming result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.000064 seconds


## 5. Branch And Bound


In [6]:
bnb_result, bnb_time = solve_baby_branch_and_bound()
print("Branch-and-bound result:", bnb_result)
print(f"Elapsed time: {bnb_time:.6f} seconds")
assert bnb_result == EXPECTED


Branch-and-bound result: {'invited': [1, 4, 5], 'names': ['Jose', 'Valentina', 'Camila'], 'total_value': 221}
Elapsed time: 0.000034 seconds


## Summary


In [7]:
rows = [
    ("Naive enumeration", naive_result, naive_time),
    ("Backtracking with pruning", backtracking_result, backtracking_time),
    ("Constraint-driven reduced search", reduced_result, reduced_time),
    ("Dynamic programming", dp_result, dp_time),
    ("Branch and Bound", bnb_result, bnb_time),
]

for method_name, result, elapsed in rows:
    print(f"{method_name:35s} value={result['total_value']:3d} invited={result['names']} time={elapsed:.6f}s")

assert all(result == EXPECTED for _, result, _ in rows)
print("\nThe optimal invitation list is Jose, Valentina and Camila, with total value 221.")


Naive enumeration                   value=221 invited=['Jose', 'Valentina', 'Camila'] time=0.000046s
Backtracking with pruning           value=221 invited=['Jose', 'Valentina', 'Camila'] time=0.000045s
Constraint-driven reduced search    value=221 invited=['Jose', 'Valentina', 'Camila'] time=0.000009s
Dynamic programming                 value=221 invited=['Jose', 'Valentina', 'Camila'] time=0.000064s
Branch and Bound                    value=221 invited=['Jose', 'Valentina', 'Camila'] time=0.000034s

The optimal invitation list is Jose, Valentina and Camila, with total value 221.
